# Brain Connectivity & GNNs for Autism Classification

Today, you'll explore how to use machine learning to predict autism spectrum disorder (ASD) from resting-state fMRI data. We will build two types of models: a classical machine learning model (SVM) based on functional connectivity, and a more advanced Graph Neural Network (GNN) model.

Don't forget to select the runtime type of this notebook to GPU.

First, let's install and import all the necessary libraries. We'll be using `nilearn` for neuroimaging data, `scikit-learn` for classical ML, and `PyTorch` with `torch_geometric` for the GNN.

In [ ]:
# Install necessary packages
!pip install nilearn
!pip install torch_geometric

In [ ]:
# Import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from nilearn import datasets, connectome, plotting
from nilearn.maskers import NiftiLabelsMasker
from nilearn.image import index_img
from nilearn.connectome import ConnectivityMeasure
from nilearn.plotting import plot_connectome, plot_matrix, find_parcellation_cut_coords

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, balanced_accuracy_score, f1_score
)
from sklearn.pipeline import Pipeline

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader, Dataset
from torch_geometric.nn import GATConv, global_mean_pool
from torch_geometric.utils import to_dense_adj

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Exercise 0: Autism and the ABIDE Dataset

Autism spectrum disorder (ASD) is characterised by qualitative impairment in social reciprocity, and by repetitive, restricted, and stereotyped behaviours/interests. Previously considered rare, ASD is now recognized to occur in more than 1% of children. The Autism Brain Imaging Data Exchange (ABIDE) dataset, which you'll be using today, includes functional and structural brain imaging data collected from laboratories around the world to accelerate our understanding of the neural bases of autism.


`nilearn` provides a convenient function to fetch a preprocessed version of this dataset, previously preprocessed by the **Preprocessed Connectomes Project (PCP)**.

The cell below will load 500 subjects from the ABIDE dataset; you can select more but I suggest you keep this number to keep download times and memory manageable for the exercises. The parameter `derivatives=['rois_aal']` will give us pre-extracted time-series extracted using the AAL atlas. Instead of downloading the full 4D fMRI images, we get time series that have already been averaged within specific brain regions defined by the AAL atlas. The parameter `quality_checked=True` filters the dataset to include only subjects who passed a set of quality control checks.

Check the documentation for the [nilearn.datasets.fetch_abide_pcp](https://nilearn.github.io/stable/modules/generated/nilearn.datasets.fetch_abide_pcp.html) function to have a better feeling of all the parameters that you can play with.

In [ ]:
# Could take ~5 minutes
abide_dataset = datasets.fetch_abide_pcp(
    n_subjects=500,
    derivatives=['rois_aal'],
    quality_checked=True
)

The downloaded data includes phenotypic information (like age, sex, and diagnosis) for each subject. The target variable `DX_GROUP` is categorical (1 for Autism, 2 for Control). We'll convert it to a binary format (1 for Autism, 0 for Control) which is more standard for binary classification tasks.

Notice that usually we say "Control" instead of "Healthy". Participants in the control group may have other conditions (e.g., anxiety, allergies, mild depression) — they're not necessarily "healthy" in every sense. What defines them as controls is that they don't meet diagnostic criteria for ASD, not that they're free of all health concerns.

# Exercise 1: Explore the dataset structure

**Task:** Inspect the returned object. Answer the following:

1. What keys does `abide_dataset` have?
2. What is the shape of the time-series for the first subject?
3. What does the phenotypic information contain?
4. What many people we have diagnosed with autism?
5. Plot the distribution of diagnosis by sex.

**Tip:** The object is a `Bunch` (dict-like). Use `abide_dataset.keys()` to see available fields. The phenotypic information is in `abide_dataset.phenotypic`.

In [ ]:
# ...

# Exercise 2: Computing Connectivity Matrices

Each subject has a time-series of shape `(T, 116)` (T time points measured across 116 AAL brain regions). To capture how regions interact, we compute a **functional connectivity matrix** (116×116) that summarises the statistical dependencies between all pairs of regions.

We will use **partial correlation** as our primary connectivity measure.



**Tasks:**
1. Compute the **partial** connectivity matrix for each subject using Nilearn's `ConnectivityMeasure`. As you should have checked already, you'll find the timeseries in `abide_dataset['rois_aal']`
3. Visualise the connectivity matrix for one ASD subject and one Control subject

In [ ]:
# ...

The code below is computing the mean connectivity matrix for ASD and Control groups separately, and checking their difference.

What do you observe, is this difference higher or lower than what you were expecting? Notice that the colourbar range of the third plot is different.

In [ ]:
tmp_labels_array = (pheno_df['DX_GROUP'].values == 2).astype(int)  # 1=control, 0=ASD
tmp_sex_array = pheno_df['SEX'].values  # 1=Male, 2=Female

mean_asd = correlation_matrices[labels == 0].mean(axis=0)
mean_ctrl = correlation_matrices[labels == 1].mean(axis=0)
diff = mean_ctrl - mean_asd

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

im0 = axes[0].imshow(mean_asd, cmap='viridis', vmin=-1, vmax=1)
axes[0].set_title('Mean Connectivity - ASD')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(mean_ctrl, cmap='viridis', vmin=-1, vmax=1)
axes[1].set_title('Mean Connectivity - Control')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(diff, cmap='viridis')#, vmin=-2, vmax=2)
axes[2].set_title('Difference (Control - ASD)')
plt.colorbar(im2, ax=axes[2], fraction=0.046)

plt.tight_layout()
plt.show()

print(f"Max absolute difference: {np.abs(diff).max():.4f}")
print(f"Mean absolute difference: {np.abs(diff).mean():.4f}")

# Exercise 3: Extracting the upper triangle features

For each connectivity matrix, extract the **upper triangle** (excluding the diagonal) as a flat feature vector, such that we can use later as a comparison with the GNN model.

Store all subjects' features in a 2D array `X_flat` of shape `(n_subjects, n_features)`


**Tip:**
 - Use `np.triu_indices(n_regions, k=1)` to get the indices of the upper triangle (k=1 excludes the diagonal).

In [ ]:
# ...

# Exercise 4: Train / Validation / Test Splits

We will use a **75/15/10** (train/validation/test) splits, **stratified by both diagnosis AND sex**. This ensures each split has similar proportions of ASD/Control subjects and male/female subjects.

**Tasks:**
1. Create a combined stratification key from diagnosis label and sex (e.g., `"0_1"` for ASD male)
2. First split: separate out 10% as the **test** set
3. Second split: from the remaining 90%, take ~16.7% as **validation** (≈15% of total)
4. Print the sizes and class distributions for each split

**Tip:**
- Use `StratifiedShuffleSplit` from sklearn.


In [ ]:
# ...
# ..., test_idx = ...
# ...
# train_idx = ...
# val_idx = ...
# ...

# Exercise 5: Baseline SVM Classifier

A classic approach in neuroimaging is to flatten the upper triangle of the connectivity matrix (which you calculated already) and feed it to a traditional ML classifier. We use an **SVM with RBF kernel** as our baseline.

**Tasks:**
1. Create a scikit-learn pipeline with `StandardScaler` + `SVC` (use `kernel='rbf'`, `C=1.0`, `probability=True`)
2. Train on the training set
3. Evaluate on the **validation set** (accuracy, balanced accuracy, F1, AUC). If at the end of the notebook you have some time, you can use the validation set to decide on different hyperparameters.
4. Report final performance on the **test set**. Plot the confusion matrix for later analysis.

In [ ]:
# X_train, y_train = X_flat[train_idx], labels[train_idx]
# X_val, y_val = X_flat[val_idx], labels[val_idx]
# X_test, y_test = X_flat[test_idx], labels[test_idx]

# svm_pipeline = Pipeline([
#    ...

# ...

# Exercise 6: Graph Construction & GAT Model

Now for the exciting part! Instead of flattening the connectivity matrix, we treat each subject's brain as a **graph**:
- **Nodes** = brain regions (116 AAL regions)
- **Edges** = connections between regions (from the connectivity matrix)
- **Node features** = spatial coordinates of each region ins space (like positional encodings!)
- **Task** = graph-level binary classification (ASD vs Control)

We use the AAL atlas coordinates as node features. This is analogous to **positional encodings** in transformers. As we saw in class, we could choose different ways to create the graph.

For speed, we start with just the 3 spatial coordinates (x, y, z) from the AAL atlas as node features. However, keep in mind that these are identical across all subjects, meaning the model can only learn from differences in graph structure and edge weights. If you have time, try enriching the node features with subject-specific connectivity statistics (e.g., node strength, degree, mean and std of correlations per region), which will give the model a much stronger signal to distinguish ASD from controls.

I will give you the code to extract the AAL atlas coordinates and normalise them between [0, 1] to save you some time. Just so you understand, the coordinates represent the centre-of-mass of each brain region in a so called "MNI space" (x, y, z in mm). We normalise them so the GNN receives inputs in a consistent range.



> <u>**NOTE:**</u> The code below might give you an error when running `fetch_atlas_aal()`. If that happens to you, download the corresponding file directly from the link you see. Then, go to the Files on the left panel, and go to `root/nilearn_data/aal_SPM12`. Inside you'll see another folder whose name is composed with many numbers and letters. Just upload the file you downloaded to that folder, and execute the cell again.

In [ ]:
atlas = datasets.fetch_atlas_aal(version='SPM12')
coordinates = find_parcellation_cut_coords(labels_img=atlas['maps'])

print(f"Atlas regions: {len(atlas['labels'])} (there is an extra Background region)")
print(f"Coordinates shape: {coordinates.shape}")
print(f"First 5 regions: {atlas['labels'][:5]}")
print(f"First 5 coordinates:\n{coordinates[:5]}")

In [ ]:
# Min-Max normalisation of coordinates
coord_min = coordinates.min(axis=0)
coord_max = coordinates.max(axis=0)
coords_normalised = (coordinates - coord_min) / (coord_max - coord_min)

print(f"Normalised coordinates shape: {coords_normalised.shape}")
print(f"Min per axis: {coords_normalised.min(axis=0)}")
print(f"Max per axis: {coords_normalised.max(axis=0)}")
print(f"\nFirst 5 normalised coordinates:\n{coords_normalised[:5]}")

# Convert to tensor (will be reused for all subjects)
node_features = torch.tensor(coords_normalised, dtype=torch.float32)
print(f"\nNode features tensor shape: {node_features.shape}")

## Exercise 6.1: Convert subjects to Pytorch Geometric graphs

**Tasks:**
1. For each subject, create a `torch_geometric.data.Data` object with:
   - `x`: node features (the normalised coordinates -same for all subjects)
   - `edge_index`: edges from the connectivity matrix (use a **threshold** to select edges, for example by keepin connections with |correlation| > 0.05)
   - `edge_attr`: edge weights (the correlation values for selected edges)
   - `y`: the subject's label
2. Store all graphs in a list
3. Print summary statistics about the graphs (mean number of edges, etc.)

**Tips:**
- Use `np.where(np.abs(matrix) > threshold)` to get edge indices
- `edge_index` in PyG expects shape `(2, num_edges)` as a `LongTensor` (COO format)

In [ ]:
# ...

# row, col = np.where(abs_corr > threshold)
# edge_index = torch.tensor(np.array([row, col]), dtype=torch.long)
# edge_weights = torch.tensor(
#    corr_matrix[row, col], dtype=torch.float32
# ).unsqueeze(1)  # shape (num_edges, 1)

# ...

## Exercise 6.2: Create data loaders

**Tasks:**
1. Split the graph list into train/val/test using the same indices from before
2. Create PyG `DataLoader` objects with `batch_size=32`

In [ ]:
# ...

## Exercise 6.3: Define the GAT model

We will now define our Graph Attention Network (GAT). A GAT uses self-attention mechanisms to learn the importance of neighboring nodes for each node in the graph.

Our model will have:
1.  Two `GATConv` layers. The first layer will use multi-head attention.
2.  A `global_mean_pool` layer to aggregate node embeddings into a single graph-level embedding.
3.  A final linear layer for classification.

The single task you have here is to fill in the `__init__` and `forward` methods of the `BrainGAT` class.

In [ ]:
class BrainGAT(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=4, dropout=0.1):
        super(BrainGAT, self).__init__()
        ### YOUR CODE HERE ###
        self.dropout = None
        self.gat1 = None
        self.gat2 = None
        self.lin = None
        ### END YOUR CODE ###

    def forward(self, x, edge_index, batch, edge_attr=None):
        # First GAT layer
        # Set return_attention_weights=True to get the learned attention scores
        x, (edge_idx1, att_weights1) = self.gat1(x, edge_index, edge_attr=edge_attr, return_attention_weights=True)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)

        # Second GAT layer
        x, (edge_idx2, att_weights2) = self.gat2(x, edge_index, edge_attr=edge_attr, return_attention_weights=True)
        x = F.elu(x)

        # Readout layer
        x = global_mean_pool(x, batch)

        # Final classifier
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.lin(x)

        return x, (edge_idx1, att_weights1), (edge_idx2, att_weights2)

# Initialize the model
model = BrainGAT(
    in_channels=3, # x, y, z coordinates
    # in_channels=7 # 3 coordinates + 4 connectivity features in case you did that part
    hidden_channels=32,
    out_channels=2, # ASD vs Control
    heads=8,
    dropout=0
).to(device)

print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

# Exercise 7: GAT Training

## Exercise 7.1: Training

**Tasks:**
1. Define an optimizer and loss function
2. Write a training loop for 100 epochs
3. Track training loss, validation loss, and validation accuracy at each epoch
4. Plot the training curves at the end

**Tips:**
- In the training loop: `for batch in train_loader: ...`
- Access `batch.x`, `batch.edge_index`, `batch.batch`, `batch.y`
- Use `model.eval()` and `torch.no_grad()` for validation
- Don't forget to save the best model based on the best validation loss

In [ ]:
# optimizer = torch.optim.Adam(...)
# criterion = nn.CrossEntropyLoss()

# ...

# if val_loss > best_val_loss:
#     best_val_loss = val_loss
#     best_model_state = {k: v.clone() for k, v in model.state_dict().items()}

# ...

# Exercise 7.2: Evaluate GAT on the test set

**Tasks:**
1. Load the best model weights
2. Evaluate on the test set
3. Report accuracy, balanced accuracy, F1, and AUC
4. Plot the confusion matrix

**Tip:**
 - Use `model.load_state_dict(best_model_state)` to load the best model.

In [ ]:
# ...